In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
os.environ["PYKEOPS_VERBOSE"] = "0"
import sys
sys.path.append(os.path.abspath("conditional-flow-matching"))
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from visualize.plots import *

In [3]:
import random
import umap
from models.modules import *
from visualize.plots import *

In [4]:
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
from scripts.run_model import *
from eval.eval import *

In [6]:
%reload_ext autoreload
%autoreload 2

In [7]:
import os
download_path = os.path.expanduser("~")
print(download_path)

/home/mingxuanzhang


In [8]:
# import scanpy as sc
# import anndata as ad
# import os
# from pathlib import Path
# from scipy.io import mmread

# base_path = Path("/home/mingxuanzhang/zebrafish/scRNAseq")

# subdirs = [d for d in base_path.iterdir() if d.is_dir() and (d / "matrix.mtx").exists()]


# adatas = []
# for subdir in subdirs:
#     timepoint = subdir.name
#     print(f"Loading: {timepoint}")
    
#     # Load raw matrix
#     X = mmread(subdir / "matrix.mtx").tocsr()
    
#     # Load genes and barcodes
#     genes_df = pd.read_csv(subdir / "genes.tsv", sep="\t", header=None)
#     barcodes_df = pd.read_csv(subdir / "barcodes.tsv", sep="\t", header=None)

#     # Transpose if needed
#     if X.shape[0] == genes_df.shape[0]:
#         X = X.T  # Convert from genes × cells → cells × genes

#     # Create AnnData
#     adata = ad.AnnData(X)
#     adata.var_names = genes_df[0].astype(str).values
#     adata.obs_names = barcodes_df[0].astype(str).values

#     # Add timepoint info
#     adata.obs["timepoint"] = timepoint

#     # Make unique cell names across timepoints (optional but recommended)
#     adata.obs_names = [f"{timepoint}_{bc}" for bc in adata.obs_names]

#     adatas.append(adata)

# # Concatenate and save
# adata_concat = ad.concat(adatas, axis=0)

In [9]:
# sc.pp.calculate_qc_metrics(
#     adata_concat, inplace=True, log1p=True
# )

In [10]:
# adata_concat = adata_concat[adata_concat.obs['n_genes_by_counts'] >= 2000, :]

In [11]:
# sc.pp.filter_genes(adata_concat, min_cells=10)

In [12]:
# adata_concat.var["mt"] = adata_concat.var_names.str.startswith("MT-")
# # ribosomal genes
# adata_concat.var["ribo"] = adata_concat.var_names.str.startswith(("RPS", "RPL"))
# # hemoglobin genes
# adata_concat.var["hb"] = adata_concat.var_names.str.contains("^HB[^(P)]")

In [13]:
# mito_genes = adata_concat.var_names.str.startswith("MT-")
# mito_gene_names = adata_concat.var_names[mito_genes]

In [14]:
# adata_concat.obs['mito_expr'] = adata_concat[:, mito_genes].X.sum(axis=1).A1

In [15]:
# percentile = 90
# threshold = np.percentile(adata_concat.obs['mito_expr'], percentile)

In [16]:
# adata_concat.obs['high_mito'] = adata_concat.obs['mito_expr'] > threshold
# adata_concat = adata_concat[~adata_concat.obs['high_mito'], :]

In [17]:
# adata_concat = sc.pp.sqrt(adata_concat, copy=True)
# adata_concat

In [18]:
# adata_concat.write_h5ad("EB.h5ad")

In [19]:
# adata.write_h5ad("EB_pca.h5ad")

In [20]:
energy_only = False

In [21]:
d = 5

In [ ]:
config = Config(
    {        
        "model_class": "metricflow",
        "score_max_epochs": 500,
        "energy_max_epochs": 2000,
        "embed_max_epochs": 2 if energy_only else 2000,
        "flow_max_epochs": 2 if energy_only else 2000,
        "lr": 1e-4,
        "dropout": 0.0,
        "pc_dim": d,
        "cond_dim": d,
        "hidden_dim": 256,
        "score_batch_size": 1024,
        "flow_batch_size": 512,
        "num_freq": 32,
        "num_layers": 4,
        "control_only": True,
        "force_cpu": False,
        "constrain": False,
        "gradient_clip_val": 10,
        "loader_batch_size": 1024,
        "warmup_steps": 0,
        # "ema_decay": None,
        "ema_decay": 1-1e-3,

        "num_workers_score": 0,
        "num_workers_flow": 0,

        "pita_steps": 3,

        "score_alpha": 0.99,
        "energy_noise_sigma": 0.05,

        "knn_interpolation": False,

        "sigma_dim": 0,

        "fast_ot": False,

        "num_sigmas": 20,
        "sigma_min": 0.1,
        "sigma_max": 0.2,

        "score_beta_min": 5.0,
        "score_beta_max": 10.0,
        "num_eigs": 3,
        
        "geo_sigma": 0.0,

        "latent_dim": 100,

        "skip": False,
        "rescale": 0.5,

        "pre_low_q": .05,
        "pre_high_q": .98,
        "low_q": .05,
        "high_q": .95,

        "weight_beta": 1.0,

        "beta_low": 1.0,
        "beta_high": 1.0,
        "gamma": 0.5,
        "margin": 1.0,

        "sigma": 0.05,
        "cfg_p_u": 0.0,
        "cfg_w": 1.0,
    }
)

In [23]:
adata, values = process_data(pc_dim=config.pc_dim, data="EB")

In [24]:
adata.uns['std'] = np.std(adata.obsm['X_pca'], axis=0, keepdims=True) #top pc to std=1
adata_raw = adata.copy()
adata.obsm['X_pca'] /= adata.uns['std']

In [25]:
adata = adata[adata.obs['timepoint'].isin([1, 2, 3])]

In [26]:
heldout_timepoint = 2
test_bool = adata.obs['timepoint'] == heldout_timepoint
adata_train = adata[~test_bool]
adata_test = adata[test_bool]

In [27]:
adata_train.obsm['X_pca'].shape

(3305, 5)

In [28]:
conditions, dataset = extract_dataset(adata_train, values)

[1, 3]


In [29]:
dataset

[(tensor([[ 1.3201, -0.5302, -0.8367,  0.1238, -0.5223],
          [ 0.8261,  0.0231,  0.3877, -0.3795, -0.0489],
          [ 0.8023,  0.2034, -0.1286, -0.2114,  0.5381],
          ...,
          [ 0.5929, -0.9067, -0.5908,  0.8898, -0.0300],
          [ 1.0876, -0.6569, -0.7620,  0.4628, -0.5163],
          [ 0.7259, -0.2028,  0.7434, -1.6256, -0.5293]]),
  tensor([[-1.1470, -0.4904, -0.8222, -1.5646,  1.7203],
          [-0.9493,  2.1083, -1.9880,  0.5425, -1.6223],
          [ 0.4408,  1.1051,  2.1330,  0.1513,  0.1264],
          ...,
          [ 0.0286,  1.2236,  0.3067,  0.9505,  0.7832],
          [-1.1362,  0.4052, -0.3208, -1.1303, -0.4404],
          [ 0.8654,  1.3215,  2.2680, -1.1927, -0.7425]]),
  1,
  3,
  'ctrl-inj')]

In [30]:
project = "EB"

In [31]:
pre_score_model, pre_energy_model, score_model, energy_model, embed_model, flow_model = run_full_model(config, project, adata, values, conditions, dataset)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Running phase score:.......


wandb: Currently logged in as: mingxuan-zhang to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/mingxuanzhang/anaconda3/lib/python3.12/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type | Params | Mode 
-------------------------------------------
0 | score_net | EMA  | 271 K  | train
-------------------------------------------
135 K     Trainable params
135 K     Non-trainable params
271 K     Total params
1.088     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/mingxuanzhang/anaconda3/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'trai

epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇█████
train_loss,▆▂▇▆▃▁▄▁▄▄▃▄█▆▄▅▆█▃▅▂▆▃▅▅▄█▂▆▂▁▄▄▅▃▃▂▅▄▅
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇██
epoch,999
train_loss,1.1372
trainer/global_step,4999


...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/mingxuanzhang/anaconda3/lib/python3.12/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type               | Params | Mode 
------------------------------------------------------------
0 | embed_net    | SimpleDenseNet     | 226 K  | train
1 | geo_net      | SinNet             | 221 K  | train
2 | energy_model | EnergyNetTrainBase | 407 K  | eval 
------------------------------------------------------------
447 K     Trainable params
407 K     Non-trainable params
855 K     Total params
3.423     Total estimated model params size (MB)
31        Mod

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇█
train_loss,█▇▅▄▄▄▄▃▄▄▃▃▃▃▂▂▃▂▃▃▂▃▂▂▂▃▁▂▂▁▁▁▁▂▂▁▁▁▂▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇████
epoch,999
train_loss,3590.40137
trainer/global_step,999


Running phase flow:.......


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/mingxuanzhang/anaconda3/lib/python3.12/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type           | Params | Mode 
-----------------------------------------------------
0 | flow_net  | SinNet         | 219 K  | train
1 | embed_net | SimpleDenseNet | 226 K  | eval 
2 | geo_net   | SinNet         | 221 K  | eval 
-----------------------------------------------------
441 K     Trainable params
226 K     Non-trainable params
667 K     Total params
2.671     Total estimated model params size (MB)
16        Modules in train mode
31        Modules in ev

epoch,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇███
train_loss,█▃▂▂▂▂▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇██
epoch,999
train_loss,0.71272
trainer/global_step,999


In [33]:
values

['ctrl-inj']

In [35]:
value = 'ctrl-inj'
t = 2
num_traj = 1000

print(predict(flow_model, adata_raw, value, conditions, num_traj, t, p=1))

6.680611610412598


In [37]:
from torch.utils.data import TensorDataset, Sampler, DataLoader
from utils.dataset import *

train_dataset = ShufflingDataset(dataset, config.flow_batch_size, conditions)
train_dataloader = DataLoader(train_dataset, batch_size = config.loader_batch_size, shuffle=True)

In [38]:
for batch in train_dataloader:
    
    paths = embed_model.sample_geodesic(batch, points=1000, ot_sample=True)
    
    break

KeyboardInterrupt: 

In [ ]:
j = 7
traj = paths[:,j]

x0 = traj[:-1]
x1 = traj[1:]
diff = x1 - x0

r = 10
fig, axs = plt.subplots(r, figsize = (8,8))
for i in range(r):
    axs[i].scatter(np.arange(diff.shape[0]), diff[:,i])
plt.show()